# Homework 1: End-to-End Reconstruction with CNNs

This homework asks you to implement a compact **end-to-end reconstruction pipeline** using the ideas developed in the lecture. You will start from clean Mayo images, generate synthetic measurements with a known operator $K$, train simple neural networks with `MSELoss`, and compare the quality of the reconstructed images. The goal is not to build the strongest possible method, but to understand the full workflow and test a few basic design choices in a controlled setting.

You should submit the completed notebook together with a short written discussion of the final questions. Keep the code readable and add brief comments only where your implementation is not obvious.

## Tasks

Work through the notebook in order. In particular, you should:

1. build a dataset pipeline for the Mayo images;
2. generate measurements of the form
   $$
   y^\delta = Kx + e,
   $$
   where $K$ is a motion-blur operator and $e$ is additive Gaussian noise;
3. implement and train both a `SimpleCNN` and a `ResCNN`;
4. compare the two models visually and, if you wish, quantitatively;
5. answer the short conceptual questions at the end.

Use `nn.MSELoss()` and generate the corrupted data **online during training**. You may work on CPU, but a GPU is recommended if available.


In [ ]:
import glob
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

here = Path.cwd().resolve()
for base in (here, here.parent):
    if (base / 'IPPy').exists():
        ippy_root = base / 'IPPy'
        break
else:
    raise FileNotFoundError('Could not locate the local IPPy package.')

operators_spec = importlib.util.spec_from_file_location('course_ippy_operators', ippy_root / 'operators.py')
operators = importlib.util.module_from_spec(operators_spec)
operators_spec.loader.exec_module(operators)


def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    try:
        if torch.backends.mps.is_available():
            return 'mps'
    except AttributeError:
        pass
    return 'cpu'


def gaussian_noise(y, noise_level):
    e = torch.randn_like(y, device=y.device)
    return e / torch.norm(e) * torch.norm(y) * noise_level


device = get_device()
torch.manual_seed(0)
print('Working device:', device)


## Implementation

Complete the code cells below. A good workflow is the following:

- first make sure the dataset and dataloaders work correctly;
- then define the motion blur operator and inspect one corrupted example;
- implement the two networks;
- train both models with the same settings;
- finally compare the reconstructions on a few test images.

Use `tqdm` to track training, inspect tensor shapes carefully, and keep the code close to the style used in the lecture notebook.


In [ ]:
class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape):
        super().__init__()
        self.data_path = data_path
        self.data_shape = data_shape
        self.fname_list = glob.glob(f'{data_path}/*/*.png')

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        # TODO:
        # 1. load the image from disk
        # 2. convert it to grayscale
        # 3. transform it into a tensor
        # 4. resize it to self.data_shape
        # 5. return the resulting tensor of shape (1, H, W)
        raise NotImplementedError


# TODO: create train_dataset, test_dataset, train_loader, test_loader
# Suggested shape: 256
# Suggested batch size: 8


# TODO: inspect one training batch
# Print:
# - batch shape
# - dtype
# - min / max intensity
# Then display one clean image from the batch.


In [ ]:
# TODO:
# 1. define K = operators.Blurring(...)
# 2. take one image x from the training loader
# 3. compute y_delta = K(x) + gaussian_noise(...)
# 4. visualize x and y_delta


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, n_filters=32, kernel_size=3):
        super().__init__()
        # TODO: implement a 3-layer CNN
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class ResCNN(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, n_filters=32, kernel_size=3):
        super().__init__()
        # TODO: implement a residual CNN
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


# TODO:
# 1. instantiate both models
# 2. print the number of trainable parameters of each model


In [ ]:
def train_model(model, train_loader, K, num_epochs=3, noise_level=0.01, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []

    # TODO:
    # complete the training loop
    # - generate y_batch on the fly from x_batch
    # - compute the loss
    # - perform backward + optimizer step
    # - track the average epoch loss
    # - use tqdm for the batch loop

    return history


# TODO:
# 1. train both models
# 2. save the histories
# 3. plot the training curves together


In [ ]:
# TODO:
# evaluate the two models on one or two test examples
# and visualize the results in a clean figure.


## Discussion

Answer the following questions in a few sentences each.

1. Which model performed better in your experiments, and why do you think that happened?
2. Did the residual architecture help? If yes, in what sense?
3. How did the noise level affect training and reconstruction quality?
4. Why is it important to generate the corruption through a known operator $K$ instead of treating the problem as a generic image-to-image task?
5. Why should one be cautious when evaluating pure end-to-end methods only through visual quality?

If you want to go a little further, try one of the following extensions: replace `MSELoss` with `L1Loss`, change the blur strength, enlarge the number of filters, or compare training with and without added noise.
